# Gapfinder Prototype

This notebook prototypes the core flow for the Gapfinder application.

## Setup

In [1]:
import os
import json
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI
from pydantic_ai import Agent
from minsearch import Index
from gitsource import chunk_documents
from youtube_transcript_api import YouTubeTranscriptApi
from IPython.display import display, Markdown

load_dotenv()
client = OpenAI()
MODEL = "gpt-4o-mini"
ytt_api = YouTubeTranscriptApi()


## Step 1: Extract & Structure Knowledge
Fetch the transcript and extract key concepts.

In [2]:
def get_video_id(url):
    """Extract video ID from YouTube URL"""
    if "v=" in url:
        return url.split("v=")[1][:11]
    elif "youtu.be/" in url:
        return url.split("youtu.be/")[1][:11]
    return url

def format_timestamp(seconds: float) -> str:
    total_seconds = int(seconds)
    hours, remainder = divmod(total_seconds, 3600)
    minutes, secs = divmod(remainder, 60)

    if hours > 0:
        return f"{hours}:{minutes:02}:{secs:02}"
    else:
        return f"{minutes}:{secs:02}"

def get_transcript(video_id):
    """Download transcript"""
    transcript = ytt_api.fetch(video_id)
    return transcript.snippets


def make_subtitles(transcript) -> str:
    lines = []

    for entry in transcript:
        ts = format_timestamp(entry.start)
        text = entry.text.replace('\n', ' ')
        lines.append(ts + ' ' + text)

    return '\n'.join(lines)

def extract_concepts(transcript_text):
    """Use LLM to extract key concepts from the transcript"""
    prompt = f"""
    You are an expert educator. Extract the most important key concepts from the following transcript of an educational video.
    Return a bulleted list of 3-5 core concepts with a brief explanation for each.
    
    Transcript:
    {transcript_text}
    """
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.2
    )
    return response.choices[0].message.content


## 2. Tool Implementaions

In [25]:
current_index = None

def process_video_and_extract_concepts(video_url: str) -> str:
    """
    Downloads the YouTube transcript, chunks/indexes it, 
    and returns the core high-level concepts taught in the video.
    """
    global current_index
    
    print(f"Processing video url to extracs video id: {video_url} ...")
    video_id = get_video_id(video_url)
    
    try:
        transcript = get_transcript(video_id)
    except Exception as e:
        return f"Error downloading transcript: {str(e)}"
        
    subtitles = make_subtitles(transcript)
    
    # Extract concepts (using first 10,000 chars to avoid massive context sizes)
    transcript_text = " ".join([s.text for s in transcript])
    print("Extracting concepts...")
    concepts = extract_concepts(transcript_text)
    # concepts = extract_concepts(transcript_text[:10000])
    display(Markdown(concepts))
    
    # Chunk and index
    print("Building search index...")
    chunks = chunk_documents([{'content': subtitles}], size=3000, step=500)
    current_index = Index(text_fields=['content'])
    current_index.fit(chunks)
    
    result = f"Video processed successfully. Core Concepts:\n{concepts}"
    print("Video processing complete.")
    return result

def search_video_transcript(search_query: str) -> str:
    """
    Performs a lexical search over the video's transcript to find specific explanations.
    """
    global current_index
    if current_index is None:
        return "Error: No video has been processed yet. Please ask the user to provide a YouTube URL first."
    
    print(f"Searching transcript for: '{search_query}'")
    results = current_index.search(search_query, num_results=5)
    return json.dumps(results, indent=2)

def evaluate_user_answer(question: str, user_answer: str, reference_context: str) -> str:
    """
    Uses the strict GapFinder rubric to grade a user's answer.
    """
    print(f"Evaluating user answer...")
    prompt = f"""
    Evaluate the user's answers and provide a markdown-formatted response with:
    - What they understood well
    - What they misunderstood or missed
    - What to revisit
    
    <QUESTION>
    {question}
    </QUESTION>

    
    <CONTEXT>
    {reference_context}
    </CONTEXT>
    
    <USER_ANSWERS>
    {user_answer}
    </USER_ANSWERS>
    """
    print(f"question: {question}")
    print(f"reference context: {reference_context}")
    print(f"user answer: {user_answer}")
    
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": "You are an expert tutor grading a student's answer."},
            {"role": "user", "content": prompt}
        ],
        temperature=0.0
    )
    return response.choices[0].message.content

In [5]:
# Test Step 1
video_url = "https://www.youtube.com/watch?v=wjZofJX0v4M" # Example: 3Blue1Brown Transformers, the tech behind LLMs
video_id = get_video_id(video_url)
print(f"Video ID: {video_id}")

Video ID: wjZofJX0v4M


In [10]:
transcript = ytt_api.fetch(video_id)

In [11]:
transcript.snippets[0]

FetchedTranscriptSnippet(text='The initials GPT stand for Generative Pretrained Transformer.', start=0.0, duration=4.56)

In [13]:
print(len(transcript))

transcript_text = ""

for s in transcript.snippets:
    # print(s.text)
    transcript_text += s.text + " "

print(transcript_text[:500])
len(transcript_text)

443
The initials GPT stand for Generative Pretrained Transformer. So that first word is straightforward enough, these are bots that generate new text. Pretrained refers to how the model went through a process of learning from a massive amount of data, and the prefix insinuates that there's more room to fine-tune it on specific tasks with additional training. But the last word, that's the real key piece. A transformer is a specific kind of neural network, a machine learning model, and it's the core i


29116

In [29]:
transcript_text_file = Path(f'transcript_text_{video_id}.txt')
transcript_text = transcript_text_file.write_text(transcript_text)

In [15]:
subtitles = make_subtitles(transcript)

In [16]:
def process_yt_transcript(video_url):
    video_url = video_url
    video_id = get_video_id(video_url)
    transcript = get_transcript(video_id)
    subtitles = make_subtitles(transcript)
    return subtitles

In [17]:
subtitles = process_yt_transcript(video_url)

In [18]:
subtitles[:500]

"0:00 The initials GPT stand for Generative Pretrained Transformer.\n0:05 So that first word is straightforward enough, these are bots that generate new text.\n0:09 Pretrained refers to how the model went through a process of learning\n0:13 from a massive amount of data, and the prefix insinuates that there's\n0:16 more room to fine-tune it on specific tasks with additional training.\n0:20 But the last word, that's the real key piece.\n0:23 A transformer is a specific kind of neural network, a machine "

In [25]:
subtitles_file = Path(f'subtitles_{video_id}.txt')
subtitles_file.write_text(subtitles)

31603

In [11]:
subtitles_file = Path(f'subtitles_{video_id}.txt')
subtitles = subtitles_file.read_text(encoding='utf-8')

transcript_text_file = Path(f'transcript_text_{video_id}.txt')
transcript_text = transcript_text_file.read_text(encoding='utf-8')

In [12]:
print(subtitles[:500])

0:00 The initials GPT stand for Generative Pretrained Transformer.
0:05 So that first word is straightforward enough, these are bots that generate new text.
0:09 Pretrained refers to how the model went through a process of learning
0:13 from a massive amount of data, and the prefix insinuates that there's
0:16 more room to fine-tune it on specific tasks with additional training.
0:20 But the last word, that's the real key piece.
0:23 A transformer is a specific kind of neural network, a machine 


In [13]:
print("\nExtracting Concepts...")
concepts = extract_concepts(transcript_text)
display(Markdown(concepts))


Extracting Concepts...


- **Generative Pretrained Transformer (GPT)**: GPT stands for Generative Pretrained Transformer, which refers to AI models designed to generate text based on learned patterns from vast datasets. The "pretrained" aspect indicates that these models undergo initial training on a large corpus of data before being fine-tuned for specific tasks.

- **Transformer Architecture**: The transformer is a type of neural network that revolutionized AI by enabling efficient processing of sequential data. It uses mechanisms like attention blocks to allow different parts of the input data to interact and influence each other, which is crucial for understanding context in language.

- **Tokenization and Embeddings**: Input data is broken down into tokens (words or subwords), which are then converted into vectors through an embedding matrix. These vectors represent the semantic meaning of the tokens in a high-dimensional space, allowing the model to capture relationships and context.

- **Attention Mechanism**: The attention mechanism is a core component of transformers that enables the model to weigh the importance of different tokens in the context of each other. This allows the model to dynamically adjust the representation of each token based on its surrounding context, enhancing understanding and generation of coherent text.

- **Probability Distribution and Softmax**: At the output stage, the model generates a probability distribution over possible next tokens using the softmax function. This function normalizes the raw output values (logits) into a valid probability distribution, allowing the model to predict the most likely next word based on the context provided.

## Chunking

In [14]:
chunks = chunk_documents(
    [{'content': subtitles}],
    size=3000,
    step=500
)

In [15]:
print('num characters:', len(chunks[0]['content']))
print('num words:', len(chunks[0]['content'].split()))
print('num docs:', len(chunks))
print('chunk:', chunks[0])

num characters: 3000
num words: 538
num docs: 59
chunk: {'start': 0, 'content': "0:00 The initials GPT stand for Generative Pretrained Transformer.\n0:05 So that first word is straightforward enough, these are bots that generate new text.\n0:09 Pretrained refers to how the model went through a process of learning\n0:13 from a massive amount of data, and the prefix insinuates that there's\n0:16 more room to fine-tune it on specific tasks with additional training.\n0:20 But the last word, that's the real key piece.\n0:23 A transformer is a specific kind of neural network, a machine learning model,\n0:27 and it's the core invention underlying the current boom in AI.\n0:31 What I want to do with this video and the following chapters is go through a\n0:35 visually-driven explanation for what actually happens inside a transformer.\n0:39 We're going to follow the data that flows through it and go step by step.\n0:43 There are many different kinds of models that you can build using transformer

In [16]:
from minsearch import Index

index = Index(text_fields=['content'])
index.fit(chunks)

In [17]:
index.docs[0]

{'start': 0,
 'content': "0:00 The initials GPT stand for Generative Pretrained Transformer.\n0:05 So that first word is straightforward enough, these are bots that generate new text.\n0:09 Pretrained refers to how the model went through a process of learning\n0:13 from a massive amount of data, and the prefix insinuates that there's\n0:16 more room to fine-tune it on specific tasks with additional training.\n0:20 But the last word, that's the real key piece.\n0:23 A transformer is a specific kind of neural network, a machine learning model,\n0:27 and it's the core invention underlying the current boom in AI.\n0:31 What I want to do with this video and the following chapters is go through a\n0:35 visually-driven explanation for what actually happens inside a transformer.\n0:39 We're going to follow the data that flows through it and go step by step.\n0:43 There are many different kinds of models that you can build using transformers.\n0:47 Some models take in audio and produce a transc

In [102]:
import random
from openai import OpenAI

client = OpenAI()

def generate_question(chunks, n_questions=10, max_chars=500):
    """
    Generiert Fragen basierend auf zufälligen Chunks.

    Args:
        chunks (list): Liste von Chunk-Dictionaries mit 'content'
        n_questions (int): Anzahl der gewünschten Fragen
        max_chars (int): Maximale Länge des Chunk-Textes im Prompt

    Returns:
        list: Liste von generierten Fragen
    """

    sampled_chunks = random.sample(chunks, min(n_questions, len(chunks)))

    questions = []

    for chunk in sampled_chunks:
        content = chunk.get("content", "")[:max_chars]

        prompt = f"""
        You will be given a text excerpt from a document.
        
        Formulate exactly ONE meaningful question that can be answered by this text.
        The question should:
        - be specific
        - clearly relate to the content
        - be well-suited for retrieval tests
        
        Text:
        <CONTEXT>
        {content}
        <CONTEXT>
        
        Frage:
        """

        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "user", "content": prompt}
            ],
            temperature=0.7
        )

        question = response.choices[0].message.content.strip()
        questions.append(question)

    return questions

In [100]:
questions = generate_question(chunks)

In [101]:
questions

['What is the purpose of the constant T mentioned in the context of the distribution used by ChatGPT?',
 'What is the main goal of the process described in the text excerpt?',
 'What is the purpose of the system prompt in developing a chatbot based on GPT-3?',
 'What technique is being used to visualize word embeddings in the text?',
 'What is the primary goal in creating a model that predicts the next word according to the text?',
 'How can one find the word for a female monarch using the concept of embeddings?',
 'What is the number of parameters in the GPT-3 model mentioned in the text?',
 'What phenomenon occurs when a model adjusts its weights during training to create word embeddings with semantic meaning?',
 'What is the context size used for training GPT-3?',
 'What effect does having one significantly larger number in the input have on the normalized output distribution?']

In [103]:
question = questions[0]
answer = 'The constant , commonly referred to as Temperature in ChatGPT and other large language models (LLMs), is a hyperparameter used to control the randomness, "creativity," or "temperature" of the probability distribution used to select the next word (token) in a generated sequence.'

In [104]:
print(question)
print(answer)

What is the purpose of the constant T mentioned in the context of the distribution used by ChatGPT?
The constant , commonly referred to as Temperature in ChatGPT and other large language models (LLMs), is a hyperparameter used to control the randomness, "creativity," or "temperature" of the probability distribution used to select the next word (token) in a generated sequence.


In [106]:
import json
from openai import OpenAI
openai_client = OpenAI()

instructions = """
You are an expert tutor. Evaluate the user's answers to the QUESTION based on the the CONTEXT.
"""

prompt_template = """
<QUESTION>
{question}
</QUESTION>

<CONTEXT>
{context}
</CONTEXT>
""".strip()

def build_prompt(question, search_results):
    context = json.dumps(search_results, indent=2)
    prompt = prompt_template.format(
        question=question,
        context=context
    ).strip()
    return prompt

def search(question):
    return index.search(question, num_results=5)

def llm(user_prompt, instructions, model='gpt-4o-mini'):
    messages = [
        {"role": "system", "content": instructions},
        {"role": "user", "content": user_prompt}
    ]

    response = openai_client.responses.create(
        model=model,
        input=messages
    )

    return response.output_text

def rag(query):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(prompt, instructions)
    return answer

In [ ]:
query = "python function definition"
usage, result = rag(query)

## Step 2: Generate Diagnostic Questions
Generate questions based on the concepts.

In [ ]:
def generate_questions(concepts):
    """Generate diagnostic questions based on concepts"""
    prompt = f"""
    Based on the following key concepts from a video, generate 3 diagnostic questions. 
    The questions should not be simple recall questions. They should be:
    1. A concept coverage question (testing understanding of the core mechanism).
    2. An "explain in your own words" prompt.
    3. An application question (transfer knowledge to a new scenario).
    
    Concepts:
    {concepts}
    
    Output the questions clearly numbered.
    """
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.7
    )
    return response.choices[0].message.content

print("Generating Questions...")
questions = generate_questions(concepts)
display(Markdown(questions))


## Step 3 & 4: User Answers & Gap Detection
Provide mock answers and detect gaps.

In [ ]:
# Step 3: Mock User Answers
mock_answers = """
1. A neural network is a machine learning model that takes some inputs, does some math, and gives an output. The weights are like importance scores.
2. The activation function is how the network decides if a neuron should fire. I think it uses a sigmoid function to squash values between 0 and 1.
3. If I change the image, the network will probably just output random numbers because it has not been trained on that specific image.
"""
display(Markdown(f"**User Answers:**\n{mock_answers}"))


In [108]:
question = questions[0]
answer = 'The constant , commonly referred to as Temperature in ChatGPT and other large language models (LLMs), is a hyperparameter used to control the randomness, "creativity," or "temperature" of the probability distribution used to select the next word (token) in a generated sequence.'

In [107]:
def detect_gaps_with_rag(question, answers):
    search_results = search(question)
    
    user_prompt = build_prompt(question, search_results)
    
    # erweitern um Antworten
    user_prompt += f"\n\n<USER_ANSWERS>\n{answers}\n</USER_ANSWERS>\n"
    
    user_prompt += """
    Evaluate the user's answers and provide:
    - What they understood well
    - What they misunderstood or missed
    - What to revisit
    """
    
    return llm(user_prompt, instructions)

In [110]:
gap_report = detect_gaps_with_rag(question, answer)

In [111]:
display(Markdown(gap_report))

### What the User Understood Well
- The user correctly identified the constant referred to as "Temperature" in the context of ChatGPT and other large language models (LLMs).
- They mentioned its role in controlling the randomness or creativity of the probability distribution used to select the next word (token), which captures the essence of how temperature affects sampling from the distribution.

### What They Misunderstood or Missed
- While the user mentioned that temperature controls "randomness," they could clarify the mechanism: a larger temperature results in a more uniform distribution, giving lower-probability words more weight, while a smaller temperature leads to higher-probability words dominating the selection process.
- They could also emphasize that in extreme cases (like T = 0), the model will always pick the highest probability token, which may lead to less diversity in the outputs.

### What to Revisit
- The user should revisit the specific effects of different temperature settings on the output distribution—how exactly higher or lower values influence the model's behavior.
- They might also want to explore the analogy of temperature with thermodynamic equations a bit more to understand why this term was chosen in the context of the model's probability distribution. 

Overall, the user's answer is strong but would benefit from additional detail and specificity regarding how temperature directly influences the behavior of the model's output.

In [ ]:
# Step 4: Gap Detection
def detect_gaps(concepts, questions, answers):
    prompt = f"""
    You are an expert tutor. Evaluate the user's answers to the diagnostic questions based on the core concepts of the video.
    
    Core Concepts:
    {concepts}
    
    Questions:
    {questions}
    
    User's Answers:
    {answers}
    
    Provide a structured report including:
    - **What they understood well:** Identify correct points in their answers.
    - **What they misunderstood or missed:** Clearly highlight the knowledge gaps or misconceptions.
    - **What to revisit:** Specifically state which concepts they need to review.
    """
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.2
    )
    return response.choices[0].message.content

print("Detecting Gaps...")
gap_report = detect_gaps(concepts, questions, mock_answers)
display(Markdown(gap_report))


## 5. Agent Instructions & Execution Loop

In [27]:
tools = [process_video_and_extract_concepts, search_video_transcript, evaluate_user_answer]

instructions = """
You are the GapFinder Agent, an expert tutor designed to help users identify their knowledge gaps when learning from long-form YouTube videos. 

Your workflow:
1. When a user provides a YouTube URL, immediately call the `process_video_and_extract_concepts` tool.
2. Once processed, present the core concepts to the user. If the user ask a question in the first prompt, 
ask questions about this topic. If the user doesn't provide any initial questions, ask if they want to focus on a 
specific concept. Start to suggest general and basic questions what to ask based on the concepts.
3. To generate diagnostic questions, DO NOT call a tool. Instead, use the concepts and (if needed) call `search_video_transcript` 
to get details, then GENERATE the questions directly in your response. 
   - A good test consists of 1 coverage question, 1 "explain in your own words" question, and 1 application question.
4. When the user answers your questions, call the `search_video_transcript` tool to find the relevant expected answers. 
Don't use your internal knowledge to answer the quetionand use only the content from `search_video_transcript`
5. Use the `evaluate_user_answer` tool to grade them.
6. Present the gap report returned by the evaluation tool clearly to the user.

Tone: Encouraging but strict. Point out exactly what they missed. Be a helpful study buddy.
"""

search_agent = Agent(
    name='search',
    model='openai:gpt-4o-mini',
    instructions=instructions,
    tools=tools
)

message_history = []
print("GapFinder Agent is ready! Type 'stop' to exit.")

while True:
    user_prompt = input('You: ')
    if user_prompt.lower().strip() == 'stop':
        break
        
    print("Agent is thinking...")
    # Pydantic AI handles the tool-calling loop automatically!
    result = await search_agent.run(user_prompt, message_history=message_history)
    
    display(Markdown(f"**ASSISTANT:**\n{result.output}"))
    
    # Save history for the next iteration
    message_history = result.all_messages()

GapFinder Agent is ready! Type 'stop' to exit.


You:  I ant to use this video to leran more about tokenization an embeddings: "https://www.youtube.com/watch?v=wjZofJX0v4M"


Agent is thinking...
Processing video url to extracs video id: https://www.youtube.com/watch?v=wjZofJX0v4M ...
Extracting concepts...


- **Generative Pretrained Transformer (GPT)**: GPT stands for Generative Pretrained Transformer, which refers to a type of AI model that generates text based on patterns learned from vast amounts of data. The "pretrained" aspect indicates that the model has undergone initial training on a broad dataset before being fine-tuned for specific tasks.

- **Transformer Architecture**: The transformer is a specific kind of neural network architecture that has revolutionized AI, particularly in natural language processing. It utilizes mechanisms like attention blocks to allow the model to weigh the relevance of different words in context, enabling it to generate coherent and contextually appropriate text.

- **Tokenization and Embeddings**: Input data is broken down into smaller units called tokens (which can be words or parts of words). Each token is then converted into a vector representation (embedding) that captures its meaning in a high-dimensional space. This process is crucial for the model to understand and manipulate language effectively.

- **Attention Mechanism**: The attention mechanism allows the model to focus on specific parts of the input data when generating predictions. It helps the model determine which words or tokens are most relevant to each other, enhancing its ability to understand context and relationships within the text.

- **Probability Distribution and Sampling**: The model generates text by predicting the next word based on the context provided. It produces a probability distribution over potential next tokens and samples from this distribution to generate coherent text. This process involves techniques like softmax to normalize outputs into a valid probability distribution, influencing the randomness and creativity of the generated text.

Building search index...
✅ Video processing complete.


...

You:  Tokenization is the process of breaking down text into smaller units called tokens, which can be words, subwords, or characters. In natural language processing (NLP), these tokens are the basic pieces that models analyze and learn from. It is important because computers cannot directly understand raw text, so tokenization provides a structured way to represent language as data. Good tokenization can improve model performance by preserving meaning and handling variations in language more effectively.


Agent is thinking...
Searching transcript for: 'tokenization'
Evaluating user answer...
question: What is the process of tokenization, and why is it essential in natural language processing?
reference context: 
user answer: Tokenization is the process of breaking down text into smaller units called tokens, which can be words, subwords, or characters. In natural language processing (NLP), these tokens are the basic pieces that models analyze and learn from. It is important because computers cannot directly understand raw text, so tokenization provides a structured way to represent language as data. Good tokenization can improve model performance by preserving meaning and handling variations in language more effectively.


**ASSISTANT:**
Your response about tokenization is solid! You correctly defined the process and highlighted its importance in natural language processing. Here's what you did well and where you can improve:

### What You Understood Well:
- You accurately described tokenization as breaking down text into smaller units (tokens) and noted its essential role in structuring language for model analysis. 
- You emphasized that good tokenization improves model performance, which shows a good grasp of its significance.

### Areas for Improvement:
- **Depth of Explanation**: It would be beneficial to elaborate further on different tokenization techniques (such as word, character, and subword tokenization) and their specific applications.
- **Challenges in Tokenization**: Consider discussing the challenges that may arise during tokenization, like handling punctuation, variations in language, and how these challenges can impact NLP tasks.

### What to Revisit:
1. **Tokenization Techniques**: Investigate the various methods of tokenization and their use cases in NLP for a more comprehensive understanding.
2. **Challenges and Considerations**: Learn about the common challenges in tokenization and strategies to address them.
3. **Real-World Applications**: Explore how tokenization is applied in tasks like sentiment analysis and machine translation to grasp its practical significance further.

Would you like to attempt another question on embeddings, or do you want to dive deeper into tokenization techniques?

request
USER: I ant to use this video to leran more about tokenization an embeddings: "https://www.youtube.com/watch?v=wjZofJX0v4M"

response
TOOL CALL: process_video_and_extract_concepts {"video_url":"https://www.youtube.com/watch?v=wjZofJX0v4M"}

request
TOOL RETURN: process_video_and_extract_concepts

response
The video covers several core concepts related to tokenization and embeddings, which are key components in understanding how models like GPT function:

1. **Generative Pretrained Transformer (GPT)**
2. **Transformer Architecture**
3. **Tokenization and Embeddings**
4. **Attention Mechanism**
5. **Probability Distribution and Sampling**

Do you have any specific questions about these topics, or would you like to focus on a particular concept? If you're unsure, I can suggest some questions to guide your understanding.

request
USER: Ask me a question about Tokenization and Embeddings

response
Great choice! Here are some questions related to Tokenization and Embeddings:

1. **Co

You:  stop


"https://www.youtube.com/watch?v=wjZofJX0v4M"

In [32]:
messages = message_history

for m in messages:
    print(m.kind)
    for p in m.parts:
        part_kind = p.part_kind
        if part_kind == 'user-prompt':
            print('USER:', p.content)
        if part_kind == 'tool-call':
            print('TOOL CALL:', p.tool_name, p.args)
        if part_kind == 'tool-return':
            print('TOOL RETURN:', p.tool_name)
        if part_kind == 'text':
            print(p.content)
    print()

request
USER: I ant to use this video to leran more about tokenization an embeddings: "https://www.youtube.com/watch?v=wjZofJX0v4M"

response
TOOL CALL: process_video_and_extract_concepts {"video_url":"https://www.youtube.com/watch?v=wjZofJX0v4M"}

request
TOOL RETURN: process_video_and_extract_concepts

response
The video covers several core concepts related to tokenization and embeddings, which are key components in understanding how models like GPT function:

1. **Generative Pretrained Transformer (GPT)**
2. **Transformer Architecture**
3. **Tokenization and Embeddings**
4. **Attention Mechanism**
5. **Probability Distribution and Sampling**

Do you have any specific questions about these topics, or would you like to focus on a particular concept? If you're unsure, I can suggest some questions to guide your understanding.

request
USER: Ask me a question about Tokenization and Embeddings

response
Great choice! Here are some questions related to Tokenization and Embeddings:

1. **Co

In [ ]:
def handle_tool_call(tool_call):
    name = tool_call.function.name
    args = json.loads(tool_call.function.arguments)
    print(f"\n🛠️  Agent called tool: {name}")
    
    if name == "process_video_and_extract_concepts":
        return process_video_and_extract_concepts(args.get("video_url", ""))
    elif name == "search_video_transcript":
        return search_video_transcript(args.get("search_query", ""))
    elif name == "evaluate_user_answer":
        return evaluate_user_answer(args.get("question", ""), args.get("user_answer", ""), args.get("reference_context", ""))
    else:
        return f"Error: Unknown tool {name}"

def run_agent(user_message, conversation_history=None):
    if conversation_history is None:
        conversation_history = [{"role": "system", "content": AGENT_INSTRUCTIONS}]
        
    conversation_history.append({"role": "user", "content": user_message})
    
    while True:
        response = client.chat.completions.create(
            model=MODEL,
            messages=conversation_history,
            tools=tools_schema
        )
        
        message = response.choices[0].message
        conversation_history.append(message)
        
        if not message.tool_calls:
            # Agent replied with text
            return message.content, conversation_history
            
        for tool_call in message.tool_calls:
            result = handle_tool_call(tool_call)
            conversation_history.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": str(result)
            })